In [1]:
# Parameters
UF = "AC"


## Análise de tendências (Mann-Kendall / Theil-Sen)

Este notebook realiza a análise de tendência temporal (Mann-Kendall + Theil-Sen) das
concentrações médias anuais de poluentes por estação, a partir dos arquivos hospedados
em `MQAr_averages/anual/{POLUENTE}/*.csv`. Para cada poluente, gera:

- `{poluente}_trend.csv` — tabela com as métricas de tendência por estação;
- `{poluente}_stations.geojson` — pontos das estações com essas métricas, usados no mapa interativo;
- `manifest_trends.json` — resumo da execução (arquivos gerados e nº de pontos por poluente).

Os arquivos são salvos localmente em `_static/preprocessed_trends/`.

> **Conexão com o relatório:** este notebook é um pré-requisito da **Seção 4.2**
> (`secao_4/secao_4.2.ipynb`) — os GeoJSONs gerados aqui alimentam o mapa interativo da
> **Figura 34** (tendências interanuais de CO, NO₂, SO₂, MP₂,₅, MP₁₀ e O₃). Execute este
> notebook primeiro, antes de abrir `secao_4.2.ipynb`.

LEIA ATENTAMENTE ANTES DE EXECUTAR ESTE  SCRIPT

1. ATUALIZAÇÃO DOS CAMINHOS:

   Este código precisa saber em qual pasta do seu computador o projeto está salvo.
   Como cada computador possui uma estrutura de pastas diferente, você deve 
   atualizar a variável de caminho bruto (quando indicado) para
   apontar para o diretório correto na sua máquina antes de rodar o script.


2. PRESERVAÇÃO DA ESTRUTURA DE PASTAS:

   Este código funciona de forma integrada com as demais pastas, scripts e
   arquivos auxiliares exatamente na ORGANIZAÇÃO fornecida no projeto.

   -> Não mova arquivos ou pastas de lugar.

   -> Não altere o nome dos diretórios ou arquivos.

   Caso a estrutura fornecida seja alterada, as importações e chamadas de dados 
   irão falhar e o código não irá funcionar.


In [2]:
import os
import re
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import geopandas as gpd
from shapely.geometry import Point
import pymannkendall as mk
from scipy.stats import theilslopes

#Configurações de entradas e saídas
# Todos os dados de entrada estão hospedados remotamente; OUTPUT_DIR é local (onde os
# CSVs/GeoJSONs consolidados são salvos) e já compatível com TRENDS_DIR em secao_4.2.ipynb.
BASE_FOLDER = "https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/"
STATIONS_FILE = "https://arquivos.lcqar.ufsc.br/data/databases/stations/Monitoramento_QAr_BR.csv"
OUTPUT_DIR = Path("../_static/preprocessed_trends")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

#Lista de poluentes para realizar a análise
POLLUTANTS = ["CO", "NO2", "SO2", "MP10", "MP25", "O3"]

#Colunas usadas na tabela de metadados das estações
ST_COL_ID = "ID_MMA_COMPLETO"
ST_COL_LAT = "LATITUDE"
ST_COL_LON = "LONGITUDE"
ST_COL_NAME = "ID_OEMA"

FILE_EXT = ".csv"
CRS_OUT = "EPSG:4326"

#Função de análise de tendências
def trend_analysis(df, col="VALOR", period=None, alpha=0.05):
    """
    Realiza análise de tendência temporal usando Mann-Kendall e Theil-Sen.
    Requer coluna 'ano' no DataFrame de entrada.
    Retorna dicionário com métricas de tendência.
    """
    if "ANO" not in df.columns:
        raise ValueError("A coluna 'ano' é obrigatória no DataFrame.")
    if col not in df.columns:
        raise ValueError(f"A coluna '{col}' não existe no DataFrame.")

    #Define período
    if period is None:
        start_year = int(df["ANO"].min())
        end_year   = int(df["ANO"].max())
    else:
        start_year, end_year = map(int, period)

    #Recorta período completo
    df_period = (
        df.loc[(df["ANO"] >= start_year) & (df["ANO"] <= end_year)]
          .sort_values("ANO")
          .copy()
    )

    #Anos inválidos (NaN no período)
    invalid_years_list = df_period.loc[df_period[col].isna(), "ANO"].astype(int).tolist()
    invalid_years_str = ", ".join(map(str, invalid_years_list)) if invalid_years_list else ""

    #Dados válidos para a análise
    df_valid = df_period.dropna(subset=[col]).copy()
    n_years_valid = int(df_valid.shape[0])

    #Defaults
    direction = "insufficient_data"
    significant = False
    p_value = np.nan
    slope = np.nan
    median = np.nan
    percent_change = np.nan

    #Condição para rodar MK + Theil-Sen se houver dados suficientes (3 anos no mínimo)
    if n_years_valid >= 3:
        x = df_valid["ANO"].values
        y = df_valid[col].values

        mk_result = mk.original_test(y, alpha=alpha)
        slope, intercept, _, _ = theilslopes(y, x)

        median = float(np.median(y))
        percent_change = float((slope / median) * 100) if median != 0 else np.nan

        if mk_result.trend == "increasing":
            direction = "increasing"
        elif mk_result.trend == "decreasing":
            direction = "decreasing"
        else:
            direction = "no trend"

        significant = bool(mk_result.p < alpha)
        p_value = float(mk_result.p)

    return {
        "start_year": start_year,
        "end_year": end_year,
        "n_valid_years": n_years_valid,
        "invalid_years": invalid_years_str,
        "direction": direction,
        "significance": significant,
        "p_value": p_value,
        "slope": slope,
        "median": median,
        "percent_change": percent_change,
    }


# Funções auxiliares para navegar diretórios remotos (o servidor expõe listagem via
# autoindex, então extraímos nomes de pastas/arquivos com regex a partir do HTML,
# já que HTTP não suporta os.listdir()/Path.iterdir() como um caminho local).
def list_remote_entries(url, pattern):
    resp = requests.get(url)
    resp.raise_for_status()
    return sorted(set(re.findall(pattern, resp.text)))

def list_remote_dirs(url):
    return list_remote_entries(url, r'href="([^"/]+)/"')

def list_remote_files(url, ext=".csv"):
    return list_remote_entries(url, r'href="([^"]+' + re.escape(ext) + r')"')


def trend_analysis_folder(folder_url, col="VALOR", period=None, file_ext=".csv"):
    """
    Aplica trend_analysis a todos os arquivos de uma pasta remota e gera um DataFrame.
    O nome do arquivo (sem extensão) vira o nome da estação na coluna 'station'.
    """
    results = []

    try:
        filenames = list_remote_files(folder_url, ext=file_ext)
    except Exception as e:
        print(f"Falha ao listar {folder_url}: {e}")
        return pd.DataFrame(results)

    for filename in filenames:
        station_name = filename.rsplit(".", 1)[0]
        file_url = folder_url + filename

        try:
            df = pd.read_csv(file_url)
        except Exception as e:
            print(f"Falha ao ler {filename}: {e}")
            continue

        try:
            res = trend_analysis(df, col=col, period=period)
            res["station"] = station_name
            res["source_file"] = filename
            results.append(res)
        except Exception as e:
            print(f"Erro ao processar {filename}: {e}")

    return pd.DataFrame(results)

def find_pollutant_folder(base_folder, pol: str):
    """
    Retorna a URL da pasta do poluente se existir, caso contrário None.
    """
    try:
        dirs = list_remote_dirs(base_folder)
    except Exception as e:
        print(f"Falha ao listar {base_folder}: {e}")
        return None
    for d in dirs:
        if pol.lower() == d.lower() or pol.lower() in d.lower():
            return base_folder + d + "/"
    return None

def read_stations_meta(stations_file):
    """
    Lê a tabela de estações e devolve DataFrame (com colunas ID_MMA_COMPLETO, LATITUDE, LONGITUDE quando disponíveis).
    """
    try:
        df = pd.read_csv(stations_file)
    except Exception as e:
        warnings.warn(f"Não foi possível ler {stations_file}: {e}")
        return pd.DataFrame(columns=[ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON])
    # garantir colunas esperadas
    for c in [ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON]:
        if c not in df.columns:
            df[c] = pd.NA
    # remover duplicatas mantendo a primeira aparição
    df = df.drop_duplicates(subset=[ST_COL_ID])
    return df[[ST_COL_ID, ST_COL_NAME, ST_COL_LAT, ST_COL_LON]]

def merge_with_station_coords(trend_df: pd.DataFrame, stations_meta: pd.DataFrame):
    """
    Faz merge simples left entre trend_df['station'] e stations_meta[ID_MMA_COMPLETO]
    Preenche apenas lat/lon que existam na tabela de estações.
    """
    if trend_df.empty:
        return trend_df
    if stations_meta.empty:
        #Sem metadados, retorna trend_df com colunas lat/lon vazias
        trend_df[ST_COL_LAT] = pd.NA
        trend_df[ST_COL_LON] = pd.NA
        return trend_df

    merged = trend_df.merge(stations_meta, how="left", left_on="station", right_on=ST_COL_ID)
    #Se as colunas vierem com nomes diferentes, garantir que lat/lon estão presentes
    if ST_COL_LAT not in merged.columns:
        merged[ST_COL_LAT] = pd.NA
    if ST_COL_LON not in merged.columns:
        merged[ST_COL_LON] = pd.NA
    return merged

def save_results_and_geojson(pol: str, merged_df: pd.DataFrame, out_dir: Path):
    """
    Salva CSV com resultados e GeoJSON com pontos (apenas para linhas com lat/lon).
    Retorna resumo para manifest.
    """
    pol_safe = pol.lower().replace(".", "").replace(" ", "_").replace(",", "")
    csv_path = out_dir / f"{pol_safe}_trend.csv"
    geojson_path = out_dir / f"{pol_safe}_stations.geojson"

    #Salvar CSV 
    try:
        merged_df.to_csv(csv_path, index=False, encoding="utf-8")
    except Exception as e:
        print(f"Falha ao salvar CSV {csv_path}: {e}")

    #Preparar GeoDataFrame apenas com estações que têm lat e lon preenchidos
    n_points = 0
    if ST_COL_LAT in merged_df.columns and ST_COL_LON in merged_df.columns:
        mask = merged_df[ST_COL_LAT].notna() & merged_df[ST_COL_LON].notna()
        pts = merged_df.loc[mask].copy()
        if not pts.empty:
            #Converter para float
            try:
                pts[ST_COL_LAT] = pts[ST_COL_LAT].astype(float)
                pts[ST_COL_LON] = pts[ST_COL_LON].astype(float)
                pts["geometry"] = [Point(xy) for xy in zip(pts[ST_COL_LON], pts[ST_COL_LAT])]
                gdf = gpd.GeoDataFrame(pts, geometry="geometry", crs=CRS_OUT)
                gdf.to_file(geojson_path, driver="GeoJSON")
                n_points = len(gdf)
            except Exception as e:
                print(f"Falha ao gerar GeoJSON para {pol}: {e}")
                geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
                n_points = 0
        else:
            #Pasta existe mas sem pontos com coords; criar geojson vazio
            geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
            n_points = 0
    else:
        #Sem colunas lat/lon; criar geojson vazio
        geojson_path.write_text(json.dumps({"type":"FeatureCollection","features":[]}, ensure_ascii=False))
        n_points = 0

    return {"pol": pol.lower(), "csv": csv_path.name, "geojson": geojson_path.name, "n_points": int(n_points)}

#Runner principal
def process_all_pollutants(base_folder=BASE_FOLDER, stations_file=STATIONS_FILE,
                           pollutants=POLLUTANTS, out_dir: Path = OUTPUT_DIR, file_ext=FILE_EXT,
                           col="VALOR", period=None):
    manifest = {"generated": []}
    print("Base folder:", base_folder)

    #Ler tabela de estações apenas uma vez
    stations_meta = read_stations_meta(stations_file)

    for pol in pollutants:
        pol_folder = find_pollutant_folder(base_folder, pol)
        if pol_folder is None:
            print(f"Pasta do poluente '{pol}' não encontrada em {base_folder}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        #Confirmar se há arquivos CSV na pasta
        csv_files = list_remote_files(pol_folder, ext=file_ext)
        if not csv_files:
            print(f"Pasta encontrada para '{pol}' ({pol_folder}), mas sem arquivos {file_ext}; pulando.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        print(f"Processando poluente '{pol}' em: {pol_folder} (arquivos: {len(csv_files)})")

        #Executar trend_analysis_folder
        trend_df = trend_analysis_folder(pol_folder, col=col, period=period, file_ext=file_ext)

        if trend_df.empty:
            print(f"Nenhum resultado válido para {pol} após análise; pulando geração de ficheiros.")
            manifest["generated"].append({"pol": pol.lower(), "csv": None, "geojson": None, "n_points": 0})
            continue

        #Merge com metadados de estações para preencher lat/lon
        merged = merge_with_station_coords(trend_df, stations_meta)

        #Salvar CSV e GeoJSON
        summary = save_results_and_geojson(pol, merged, out_dir)
        manifest["generated"].append(summary)
        print(f"{pol}: CSV -> {summary['csv']}, GeoJSON -> {summary['geojson']} (pontos: {summary['n_points']})")

    #Salvar manifest
    manifest_path = out_dir / "manifest_trends.json"
    try:
        with open(manifest_path, "w", encoding="utf-8") as fh:
            json.dump(manifest, fh, ensure_ascii=False, indent=2)
        print("Manifest salvo em:", manifest_path)
    except Exception as e:
        print("Falha ao salvar manifest:", e)

    return manifest

#Execução
if __name__ == "__main__":
    manifest = process_all_pollutants()
    print("Concluído. Manifest:", manifest)


Base folder: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/


Processando poluente 'CO' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/CO/ (arquivos: 194)


CO: CSV -> co_trend.csv, GeoJSON -> co_stations.geojson (pontos: 174)
Processando poluente 'NO2' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/NO2/ (arquivos: 254)


NO2: CSV -> no2_trend.csv, GeoJSON -> no2_stations.geojson (pontos: 224)
Processando poluente 'SO2' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/SO2/ (arquivos: 202)


SO2: CSV -> so2_trend.csv, GeoJSON -> so2_stations.geojson (pontos: 183)
Processando poluente 'MP10' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/MP10/ (arquivos: 307)


MP10: CSV -> mp10_trend.csv, GeoJSON -> mp10_stations.geojson (pontos: 283)
Processando poluente 'MP25' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/MP25/ (arquivos: 220)


MP25: CSV -> mp25_trend.csv, GeoJSON -> mp25_stations.geojson (pontos: 182)
Processando poluente 'O3' em: https://arquivos.lcqar.ufsc.br/data/databases/stations/MQAr_averages/anual/O3/ (arquivos: 256)


O3: CSV -> o3_trend.csv, GeoJSON -> o3_stations.geojson (pontos: 231)
Manifest salvo em: ../_static/preprocessed_trends/manifest_trends.json
Concluído. Manifest: {'generated': [{'pol': 'co', 'csv': 'co_trend.csv', 'geojson': 'co_stations.geojson', 'n_points': 174}, {'pol': 'no2', 'csv': 'no2_trend.csv', 'geojson': 'no2_stations.geojson', 'n_points': 224}, {'pol': 'so2', 'csv': 'so2_trend.csv', 'geojson': 'so2_stations.geojson', 'n_points': 183}, {'pol': 'mp10', 'csv': 'mp10_trend.csv', 'geojson': 'mp10_stations.geojson', 'n_points': 283}, {'pol': 'mp25', 'csv': 'mp25_trend.csv', 'geojson': 'mp25_stations.geojson', 'n_points': 182}, {'pol': 'o3', 'csv': 'o3_trend.csv', 'geojson': 'o3_stations.geojson', 'n_points': 231}]}
